# HubbardBath 03 — Open-system thermalisation

Milestone 02 told us what thermal equilibrium looks like. Milestone 03 asks a different question: **how does the interacting fermionic system get there?**

We start from a non-equilibrium doublon state and evolve the density matrix under a finite-dimensional detailed-balance Lindblad generator.


## 1. Lindblad dynamics

We use

$$\dot\rho = -i[H,\rho] + \sum_k\left(L_k\rho L_k^\dagger - \frac12\{L_k^\dagger L_k,\rho\}\right).$$

For every bath-driven transition with Bohr frequency $\omega>0$, the rates obey

$$\frac{\gamma_\uparrow}{\gamma_\downarrow}=e^{-\beta\omega}.$$

That detailed-balance condition makes the Gibbs populations stationary. We explicitly test stationarity numerically rather than assuming it.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'HubbardBath']
for candidate in candidates:
    if (candidate / 'src' / 'open_systems.py').exists():
        repo_root = candidate.resolve()
        break
else:
    raise FileNotFoundError('Run this notebook from the HubbardBath repository.')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.hubbard import project_to_particle_sector, two_site_hubbard_hamiltonian
from src.open_systems import (
    build_thermal_liouvillian,
    density_matrix_derivative,
    doublon_initial_state,
    evolve_density_matrix,
    trace_distance,
)
from src.thermal import (
    average_double_occupancy_operator,
    expectation,
    nearest_neighbor_spin_z_correlation_operator,
    project_operator_to_sector,
)


## 2. Build the interacting system and thermal bath

The reference case uses $U/t=4$ and $\beta t=1$. Bath couplings act through local charge and spin fluctuations while preserving the total particle number $N=2$.


In [ ]:
U = 4.0
beta = 1.0
rate_scale = 0.2

H_full = two_site_hubbard_hamiltonian(t=1.0, U=U)
H_half, _ = project_to_particle_sector(H_full, 4, 2)

L, jumps, rho_beta = build_thermal_liouvillian(
    H_half, beta=beta, rate_scale=rate_scale
)

print('Hilbert-space dimension:', H_half.shape[0])
print('Liouvillian dimension:', L.shape)
print('Jump operators:', len(jumps))


## 3. Verify the Gibbs state is stationary

A thermal bath model is not useful to us unless the intended Gibbs state is actually a fixed point.


In [ ]:
residual = np.linalg.norm(density_matrix_derivative(L, rho_beta))
print(f'||L(rho_beta)|| = {residual:.3e}')
assert residual < 1e-10


## 4. Start far from equilibrium

We place both electrons on site 0. For repulsive $U>0$, this doublon configuration is energetically costly and gives us a clear non-equilibrium starting point.


In [ ]:
rho0 = doublon_initial_state(site=0)
print('Initial trace distance to Gibbs state:', trace_distance(rho0, rho_beta))


## 5. Watch the system thermalise


In [ ]:
times = np.linspace(0.0, 160.0, 321)
states = evolve_density_matrix(L, rho0, times)
distances = np.array([trace_distance(rho, rho_beta) for rho in states])

plt.figure(figsize=(7, 4.5))
plt.semilogy(times, distances)
plt.xlabel(r'Time $t$')
plt.ylabel(r'$D(\rho(t),\rho_\beta)$')
plt.title(r'Thermalisation toward the Gibbs state: $U/t=4$, $\beta t=1$')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 6. Follow material observables during relaxation

Trace distance tells us whether the full density matrix is approaching equilibrium. We also watch quantities with direct Hubbard-model meaning.


In [ ]:
D_half = project_operator_to_sector(average_double_occupancy_operator(), 4, 2)
Czz_half = project_operator_to_sector(nearest_neighbor_spin_z_correlation_operator(), 4, 2)

double_occupancy = np.array([expectation(rho, D_half) for rho in states])
spin_corr = np.array([expectation(rho, Czz_half) for rho in states])
energy = np.array([expectation(rho, H_half) for rho in states])

plt.figure(figsize=(7, 4.5))
plt.plot(times, double_occupancy, label='double occupancy')
plt.plot(times, spin_corr, label=r'$\langle S^z_0 S^z_1\rangle$')
plt.xlabel(r'Time $t$')
plt.ylabel('Observable')
plt.title('Relaxation of Hubbard observables')
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 7. What comes next

Milestone 03 establishes a validated thermalising open-system model. Milestone 04 will stop looking at one trajectory and ask the research question: **how do interaction strength and temperature change the Liouvillian spectral gap and the thermal mixing time?**
